In [4]:
!pip install optuna xgboost

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
!pip install pandas
!pip install -U scikit-learn   

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd 

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 2026
SEEDS = [2026 , 3407]
N_FOLDS = 5
TARGET_COL = "Irrigation_Need"
ID_COL = 'id'
N_CLASSES = 3

USE_GPU = True
USE_OPTUNA_CLASS_WEIGHTS = True
OPTUNA_TRIALS = 120

def seed_everything(seed : int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)
    
seed_everything(SEED)
print("Config loaded")

Config loaded


In [ ]:
DATA_DIR = Path(".")

train = pd.read_csv("playground-series-s6e4/train.csv")
test = pd.read_csv("playground-series-s6e4/test.csv")
sub = pd.read_csv("playground-series-s6e4/sample_submission.csv")

print("train:", train.shape)
print("test :", test.shape)

# Target mapping
label2idx = {"Low": 0, "Medium": 1, "High": 2}
idx2label = {v: k for k, v in label2idx.items()}

train[TARGET_COL] = train[TARGET_COL].map(label2idx).astype("int8")

test_ids = test[ID_COL].copy()
y = train[TARGET_COL].copy()

X_train_raw = train.drop(columns=[TARGET_COL]).copy()
X_test_raw = test.copy()

if ID_COL in X_train_raw.columns:
    X_train_raw = X_train_raw.drop(columns=[ID_COL])
if ID_COL in X_test_raw.columns:
    X_test_raw = X_test_raw.drop(columns=[ID_COL])

print("y distribution:")
print(y.value_counts(normalize=True).sort_index())

train: (630000, 21)
test : (270000, 20)
y distribution:
Irrigation_Need
0    0.587170
1    0.379483
2    0.033348
Name: proportion, dtype: float64


In [ ]:
from pandas.api.types import is_object_dtype, is_string_dtype, is_categorical_dtype

# Based exactly on the top performing notebook pss6e4-xgb-cv-0-979805
def FE(df, M): 
    out = df.copy()
    for c in NUMS_BASE:
        for k in range(-4, 3):
            out[f"{c}_digit{k}"] = (out[c] // (10**k) % 10).astype('int8')

        if M[c] < 10:
            out[c] = out[c].round(3)
        elif M[c] < 100:
            out[c] = out[c].round(2)
        else:
            out[c] = out[c].round(1)
    return out 

def get_base_cols(df):
    cats = [c for c in df.columns if is_object_dtype(df[c]) or is_string_dtype(df[c]) or is_categorical_dtype(df[c])]
    nums = [c for c in df.columns if c not in cats]
    return cats, nums

CATS_BASE, NUMS_BASE = get_base_cols(X_train_raw)

M_vals = X_train_raw[NUMS_BASE].max()
X_train_fe = FE(X_train_raw, M_vals)
X_test_fe  = FE(X_test_raw, M_vals)

DROP = [c for c in X_test_fe.columns if X_test_fe[c].nunique() == 1]
print(f"Dropping constant cols: {DROP}")
X_train_fe.drop(columns=DROP, inplace=True)
X_test_fe.drop(columns=DROP, inplace=True)

CATEGORY = CATS_BASE + [c for c in X_test_fe.columns if 'digit' in c]

for c in CATEGORY:
    freq = X_train_fe[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    X_train_fe[c] = X_train_fe[c].map(lambda x: mapping.get(x, mapping_default))
    X_test_fe[c] = X_test_fe[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + [c for c in NUMS_BASE if c in X_test_fe.columns]

class OrderedTE():
    def __init__(self, a=1):
        self.a = a
        
    def fit(self, train, category_cols=[], target_col='target'):
        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols
        
        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        
        for c in self.category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(np.int8)
                
                df = train[[c]].copy()
                df['y'] = y_binary.values
                df['cnt'] = 1
                
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']
                
                smooth_prior = self.a * self.global_prior_[k]
                
                te_col = f'{c}_TE_cls{cls}'
                df[te_col] = (df['cum_sum'] + smooth_prior) / (df['cum_cnt'] + self.a)
                df.loc[df['cum_cnt'] == -1, te_col] = self.global_prior_[k]  # handle if initial condition varies
                
                self.train[te_col] = df[te_col].astype(np.float32).values
                
                stats_df = df.groupby(c)['y'].agg(['count', 'sum']).reset_index()
                stats_df.columns = [c, f'{c}_count_cls{cls}', f'{c}_sum_cls{cls}']
                stats_df[f'{c}_prior_cls{cls}'] = self.global_prior_[k]
                stats_list.append(stats_df)
            
            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how='outer')
            setattr(self, f'{c}_stats', combined_stats)
            
        return self.train
    
    def transform(self, test):
        out = test.copy()
        for c in self.category_cols:
            stats_df = getattr(self, f'{c}_stats')
            out = out.merge(stats_df, on=c, how='left')
            
            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                count_col = f'{c}_count_cls{cls}'
                sum_col = f'{c}_sum_cls{cls}'
                prior_col = f'{c}_prior_cls{cls}'
                
                if count_col in out.columns:
                    numer = out[sum_col] + self.a * out[prior_col]
                    denom = out[count_col] + self.a
                    out[te_col] = numer / denom
                    out[te_col] = out[te_col].fillna(out[prior_col])
                    out.drop(columns=[count_col, sum_col, prior_col], inplace=True)
                else:
                    out[te_col] = self.global_prior_[k]
            
            out[te_col] = out[te_col].astype(np.float32)
        return out

print("Base train shape:", X_train_fe.shape)
print("Base test shape :", X_test_fe.shape)
print("cats:", len(CATEGORY), "nums:", len(NUMS_BASE))

Dropping constant cols: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_Moisture_digit2', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Temperature_C_digit2', 'Humidity_digit2', 'Sunlight_Hours_digit2', 'Wind_Speed_kmh_digit2', 'Field_Area_hectare_digit2']
Base train shape: (630000, 84)
Base test shape : (270000, 84)
cats: 73 nums: 11


In [ ]:
#Hyper parameter tuning 
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import copy
import gc

# Define sample weights globally so Optuna objectives can use them
unique, counts = np.unique(y, return_counts=True)
weights_dict = {cls: (len(y) / len(unique)) / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([weights_dict[lbl] for lbl in y])

def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 3),
        'learning_rate': trial.suggest_float('learning_rate', 0.018, 0.028),
        'n_estimators': trial.suggest_int('n_estimators', 2900, 3200),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 4),
        'subsample': trial.suggest_float('subsample', 0.62, 0.79),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.45, 0.59),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 6),
        'reg_lambda': trial.suggest_float('reg_lambda', 7.0, 8.5),
        'gamma': trial.suggest_float('gamma', 0, 0.035),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.75),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 0.55),
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'random_state': SEED,
        'n_jobs': -1
    }
    
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    
    for tr_idx, va_idx in kf.split(X_train_fe, y):
        X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
        w_tr = sample_weights[tr_idx]

        te = OrderedTE(a=1)
        tr_df = pd.concat([X_tr, y_tr], axis=1)
        tr_df['w'] = w_tr
        te_train = pd.concat([
            te.fit(tr_df.sample(frac=1, random_state=SEED+i), category_cols=FEATURES, target_col=TARGET_COL) 
            for i in range(1) # Faster for optuna
        ])
        
        X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'])
        y_tr_enc = te_train[TARGET_COL].values
        w_tr_enc = te_train['w'].values
        
        X_va_enc = te.transform(X_va)
        cols_to_drop = [c for c in CATEGORY if c in X_tr_enc.columns]
        X_tr_enc.drop(columns=cols_to_drop, inplace=True)
        X_va_enc.drop(columns=cols_to_drop, inplace=True)
        
        for c in X_tr_enc.columns:
            if str(X_tr_enc[c].dtype) != 'object' and str(X_tr_enc[c].dtype) != 'category':
                X_tr_enc[c] = pd.to_numeric(X_tr_enc[c], downcast='float')
                X_va_enc[c] = pd.to_numeric(X_va_enc[c], downcast='float')
        
        model = XGBClassifier(**params)
        model.fit(X_tr_enc, y_tr_enc, sample_weight=w_tr_enc, verbose=False)
        va_p = model.predict_proba(X_va_enc)
        scores.append(balanced_accuracy_score(y_va, va_p.argmax(axis=1)))
        
        del model, X_tr_enc, X_va_enc
        gc.collect()
        
    return np.mean(scores)

# study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
# # We will just run a few trials to save time. Change n_trials to 30-50 for actual tuning
# study_xgb.optimize(objective_xgb, n_trials=15) 
# best_xgb_params = study_xgb.best_params
# best_xgb_params['tree_method'] = 'hist'
# best_xgb_params['device'] = 'cuda' if USE_GPU else 'cpu'
# best_xgb_params['random_state'] = SEED
# print(f"Best XGBParams: {best_xgb_params}")

m_1={'max_depth': 3, 'learning_rate': 0.02203115097882011, 'n_estimators': 3007, 'min_child_weight': 4, 'subsample': 0.6845485572360555, 'colsample_bytree': 0.5421743930210993, 'reg_alpha': 0.0003235032301833287, 'reg_lambda': 7.855474172043545, 'gamma': 0.011985864186230821, 'colsample_bylevel': 0.6746011275030865, 'colsample_bynode': 0.43025554234022867}
m_2={'max_depth': 3, 'learning_rate': 0.01830593067542217, 'n_estimators': 2921, 'min_child_weight': 3, 'subsample': 0.6964345424662757, 'colsample_bytree': 0.5235933600772913, 'reg_alpha': 0.0004392137631307963, 'reg_lambda': 7.2237506996100285, 'gamma': 0.0023448219748747005, 'colsample_bylevel': 0.6705676719919281, 'colsample_bynode': 0.5023866224759387, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026}
m_3={'max_depth': 3, 'learning_rate': 0.021024765896195494, 'n_estimators': 3038, 'min_child_weight': 3, 'subsample': 0.734113269152175, 'colsample_bytree': 0.5048175102712853, 'reg_alpha': 5.703698690455624e-05, 'reg_lambda': 7.842840833803608, 'gamma': 0.018185731513283104, 'colsample_bylevel': 0.7041739619923931, 'colsample_bynode': 0.5385052123065428, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026}
best_xgb_params=m_3

In [ ]:
#Feature important snapshot 

from xgboost import XGBClassifier

# Build a quick encoded training matrix for importance estimation
te_fi = OrderedTE(a=1)
fi_df = pd.concat([X_train_fe.copy(), y.copy()], axis=1)
te_train_fi = te_fi.fit(fi_df.sample(frac=1, random_state=SEED), category_cols=FEATURES, target_col=TARGET_COL)

X_fi = te_train_fi.drop(columns=[TARGET_COL]).copy()
cols_to_drop_fi = [c for c in CATEGORY if c in X_fi.columns]
X_fi.drop(columns=cols_to_drop_fi, inplace=True)

for c in X_fi.columns:
    if str(X_fi[c].dtype) != 'object' and str(X_fi[c].dtype) != 'category':
        X_fi[c] = pd.to_numeric(X_fi[c], downcast='float')

fi_params = best_xgb_params.copy()
fi_params['n_estimators'] = min(int(fi_params.get('n_estimators', 2000)), 1200)
fi_params['random_state'] = SEED

fi_model = XGBClassifier(**fi_params)
fi_model.fit(X_fi, y)

feature_importance_df = pd.DataFrame({
    'feature': X_fi.columns,
    'importance': fi_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

TOP_N_MAIN = min(140, len(feature_importance_df))
DROP_BOTTOM_N = min(max(20, int(0.30 * len(feature_importance_df))), len(feature_importance_df) - 1)

keep_main_features = feature_importance_df['feature'].head(TOP_N_MAIN).tolist()
drop_low_importance_features = feature_importance_df['feature'].tail(DROP_BOTTOM_N).tolist()

print('Total encoded features:', len(feature_importance_df))
print('Keeping (main):', len(keep_main_features))
print('Dropping low-importance:', len(drop_low_importance_features))
print('Top 20 features:')
print(feature_importance_df.head(20))

# cleanup

del fi_model, te_fi, te_train_fi, X_fi, fi_df
gc.collect()

Total encoded features: 263
Keeping (main): 140
Dropping low-importance: 78
Top 20 features:
                                   feature  importance
0   Previous_Irrigation_mm_digit-3_TE_cls2    0.004190
1            Organic_Carbon_digit0_TE_cls0    0.004189
2            Wind_Speed_kmh_digit1_TE_cls2    0.004132
3                  Soil_pH_digit-3_TE_cls0    0.004121
4             Temperature_C_digit1_TE_cls1    0.004093
5                            Soil_Moisture    0.004093
6           Sunlight_Hours_digit-2_TE_cls1    0.004091
7                    Temperature_C_TE_cls0    0.004085
8            Wind_Speed_kmh_digit1_TE_cls1    0.004062
9            Soil_Moisture_digit-1_TE_cls0    0.004057
10      Field_Area_hectare_digit-3_TE_cls2    0.004042
11       Field_Area_hectare_digit0_TE_cls0    0.004040
12  Previous_Irrigation_mm_digit-1_TE_cls1    0.004039
13       Field_Area_hectare_digit0_TE_cls2    0.004032
14          Sunlight_Hours_digit-2_TE_cls2    0.004028
15                Humidity_

247

In [ ]:
#Hyperparameter tuning version A 
def build_xgb_variant_a(df):
    out = df.copy()

    # Explicitly remove low-importance features identified in the previous cell
    cols_drop = [c for c in drop_low_importance_features if c in out.columns]
    if cols_drop:
        out = out.drop(columns=cols_drop)

    # Interaction features on top-ranked numeric columns
    num_cols = [c for c in keep_main_features if c in out.columns and str(out[c].dtype) != 'object' and str(out[c].dtype) != 'category']
    top_num = num_cols[:8]

    for i in range(min(4, len(top_num))):
        for j in range(i + 1, min(6, len(top_num))):
            c1, c2 = top_num[i], top_num[j]
            out[f'va_mul_{i}_{j}'] = (out[c1] * out[c2]).astype(np.float32)

    if len(top_num) >= 3:
        out['va_row_mean_top'] = out[top_num].mean(axis=1).astype(np.float32)
        out['va_row_std_top'] = out[top_num].std(axis=1).fillna(0).astype(np.float32)

    return out


def objective_xgb_a(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.05, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 2800, 3400),
        'min_child_weight': trial.suggest_int('min_child_weight', 4, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 0.75),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.45, 0.65),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 5e-2, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.1, 0.3),
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'random_state': SEED,
        'n_jobs': -1
    }


    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED + 11)
    scores = []

    for tr_idx, va_idx in kf.split(X_train_fe, y):
        X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
        w_tr = sample_weights[tr_idx]

        te = OrderedTE(a=1)
        tr_df = pd.concat([X_tr, y_tr], axis=1)
        tr_df['w'] = w_tr
        te_train = te.fit(tr_df.sample(frac=1, random_state=SEED), category_cols=FEATURES, target_col=TARGET_COL)

        X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'])
        y_tr_enc = te_train[TARGET_COL].values
        w_tr_enc = te_train['w'].values

        X_va_enc = te.transform(X_va)

        cols_to_drop = [c for c in CATEGORY if c in X_tr_enc.columns]
        X_tr_enc.drop(columns=cols_to_drop, inplace=True)
        X_va_enc.drop(columns=cols_to_drop, inplace=True)

        X_tr_var = build_xgb_variant_a(X_tr_enc)
        X_va_var = build_xgb_variant_a(X_va_enc)

        model = XGBClassifier(**params)
        model.fit(X_tr_var, y_tr_enc, sample_weight=w_tr_enc, verbose=False)

        va_p = model.predict_proba(X_va_var)
        scores.append(balanced_accuracy_score(y_va, va_p.argmax(axis=1)))

        del model, X_tr_enc, X_va_enc, X_tr_var, X_va_var
        gc.collect()

    return float(np.mean(scores))


study_xgb_a = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED + 11))
study_xgb_a.optimize(objective_xgb_a, n_trials=15)

# best_xgb_a_params = study_xgb_a.best_params
# best_xgb_a_params['tree_method'] = 'hist'
# best_xgb_a_params['device'] = 'cuda' if USE_GPU else 'cpu'
# best_xgb_a_params['random_state'] = SEED
# best_xgb_a_params['n_jobs'] = -1

print(f"Best XGB Variant-A Params: {best_xgb_a_params}")
m3_3={'max_depth': 2, 'learning_rate': 0.028984821230022537, 'n_estimators': 2833, 'min_child_weight': 4, 'subsample': 0.6507812654925471, 'colsample_bytree': 0.6175093182232598, 'reg_alpha': 0.030469549379944248, 'reg_lambda': 5.997320655447417, 'gamma': 0.28542436792373316, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026, 'n_jobs': -1}
# I got 0.9734 using the bestest TE fetaures

best_xgb_a_params=m3_3
best_xgb_a_params['tree_method'] = 'hist'
best_xgb_a_params['device'] = 'cuda' if USE_GPU else 'cpu'
best_xgb_a_params['random_state'] = SEED
best_xgb_a_params['n_jobs'] = -1


NameError: name 'optuna' is not defined

In [ ]:
#Hyperparameter tuning 
from xgboost import XGBClassifier


def build_xgb_variant_b(df):
    out = df.copy()

    # Keep a broad set of important features so performance does not collapse
    keep_cols = [c for c in keep_main_features if c in out.columns]

    # Safety fallback in case feature list is unavailable or too small
    if len(keep_cols) < 40:
        keep_cols = list(out.columns)

    # Keep a larger slice and explicitly remove weakest features only
    keep_cols = keep_cols[:220]
    out = out[keep_cols].copy()

    low_cols = [c for c in drop_low_importance_features if c in out.columns]
    if len(low_cols) > 0:
        # Drop only a small fraction from the low-importance list
        out = out.drop(columns=low_cols[: min(25, len(low_cols))])

    num_cols = [
        c for c in out.columns
        if str(out[c].dtype) != 'object' and str(out[c].dtype) != 'category'
    ]
    top_num = num_cols[:8]

    # Mild nonlinear signals on very top numeric features
    for c in top_num[:4]:
        x = pd.to_numeric(out[c], errors='coerce').fillna(0.0).astype(np.float32)
        out[f'vb_sq_{c}'] = np.square(x).astype(np.float32)

    if len(top_num) >= 3:
        out['vb_row_mean_top'] = out[top_num].mean(axis=1).astype(np.float32)
        out['vb_row_std_top'] = out[top_num].std(axis=1).fillna(0).astype(np.float32)

    return out


def objective_xgb_b(trial):
    # Keep Variant-B near strong baseline search space
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.012, 0.028),
        'n_estimators': trial.suggest_int('n_estimators', 2600, 3400),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 6),
        'subsample': trial.suggest_float('subsample', 0.65, 0.90),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.50, 0.80),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 5e-3, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 2.0, 10.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.06),
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'random_state': SEED,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }

    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED + 22)
    scores = []

    for tr_idx, va_idx in kf.split(X_train_fe, y):
        X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
        w_tr = sample_weights[tr_idx]

        te = OrderedTE(a=1)
        tr_df = pd.concat([X_tr, y_tr], axis=1)
        tr_df['w'] = w_tr
        te_train = te.fit(
            tr_df.sample(frac=1, random_state=SEED + 1),
            category_cols=FEATURES,
            target_col=TARGET_COL
        )

        X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'])
        y_tr_enc = te_train[TARGET_COL].values
        w_tr_enc = te_train['w'].values

        X_va_enc = te.transform(X_va)

        cols_to_drop = [c for c in CATEGORY if c in X_tr_enc.columns]
        X_tr_enc.drop(columns=cols_to_drop, inplace=True)
        X_va_enc.drop(columns=cols_to_drop, inplace=True)

        X_tr_var = build_xgb_variant_b(X_tr_enc)
        X_va_var = build_xgb_variant_b(X_va_enc)

        # Ensure numeric dtypes and aligned columns for robust fitting
        for c in X_tr_var.columns:
            if str(X_tr_var[c].dtype) != 'object' and str(X_tr_var[c].dtype) != 'category':
                X_tr_var[c] = pd.to_numeric(X_tr_var[c], downcast='float')
                X_va_var[c] = pd.to_numeric(X_va_var[c], downcast='float')

        X_va_var = X_va_var.reindex(columns=X_tr_var.columns, fill_value=0)

        model = XGBClassifier(**params)
        model.fit(X_tr_var, y_tr_enc, sample_weight=w_tr_enc, verbose=False)

        va_p = model.predict_proba(X_va_var)
        scores.append(balanced_accuracy_score(y_va, va_p.argmax(axis=1)))

        del model, X_tr_enc, X_va_enc, X_tr_var, X_va_var
        gc.collect()

    return float(np.mean(scores))


study_xgb_b = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED + 22))
# Keep trial failures from stopping the full tuning loop
study_xgb_b.optimize(objective_xgb_b, n_trials=15, catch=(Exception,))

# best_xgb_b_params = study_xgb_b.best_params
# best_xgb_b_params['tree_method'] = 'hist'
# best_xgb_b_params['device'] = 'cuda' if USE_GPU else 'cpu'
# best_xgb_b_params['random_state'] = SEED
# best_xgb_b_params['n_jobs'] = -1
# best_xgb_b_params['eval_metric'] = 'mlogloss'

m2_1={'max_depth': 4, 'learning_rate': 0.02281285291900719, 'n_estimators': 3369, 'min_child_weight': 2, 'subsample': 0.8584463266656952, 'colsample_bytree': 0.7909734997323306, 'reg_alpha': 0.00033269837473256587, 'reg_lambda': 4.3657215227333985, 'gamma': 0.0189127585639387, 'tree_method': 'hist', 'device': 'cuda', 'random_state': 2026, 'n_jobs': -1, 'eval_metric': 'mlogloss'}
best_xgb_b_params=m2_1
best_xgb_b_params['tree_method'] = 'hist'
best_xgb_b_params['device'] = 'cuda' if USE_GPU else 'cpu'
best_xgb_b_params['random_state'] = SEED
best_xgb_b_params['n_jobs'] = -1
best_xgb_b_params['eval_metric'] = 'mlogloss'
# i got best 0.88 in m2_1 using less important fetaure to train model on those fetaures which is not much used in previous model

print(f"Best XGB Variant-B Params: {best_xgb_b_params}")



In [ ]:
# -----------------------------
# CV training: XGBoost Trio (Base + Variant-A + Variant-B)
# -----------------------------
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type) != 'category':
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df


xgb_base_params = best_xgb_params.copy()
xgb_a_params = best_xgb_a_params.copy()
xgb_b_params = best_xgb_b_params.copy()

n_folds = 5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

oof_xgb_base = np.zeros((len(y), N_CLASSES), dtype=np.float32)
oof_xgb_a = np.zeros((len(y), N_CLASSES), dtype=np.float32)
oof_xgb_b = np.zeros((len(y), N_CLASSES), dtype=np.float32)

test_xgb_base = np.zeros((len(X_test_fe), N_CLASSES), dtype=np.float32)
test_xgb_a = np.zeros((len(X_test_fe), N_CLASSES), dtype=np.float32)
test_xgb_b = np.zeros((len(X_test_fe), N_CLASSES), dtype=np.float32)

print('Starting XGBoost Trio CV...')

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_fe, y), start=1):
    print(f"\n--- Fold {fold}/{n_folds} ---")

    X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
    y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
    w_tr = sample_weights[tr_idx]

    te = OrderedTE(a=1)
    tr_df = pd.concat([X_tr, y_tr], axis=1)
    tr_df['w'] = w_tr

    te_train = pd.concat([
        te.fit(tr_df.sample(frac=1, random_state=SEED + i), category_cols=FEATURES, target_col=TARGET_COL)
        for i in range(2)
    ])

    X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'])
    y_tr_enc = te_train[TARGET_COL].values
    w_tr_enc = te_train['w'].values

    X_va_enc = te.transform(X_va)
    X_te_enc = te.transform(X_test_fe)

    cols_to_drop = [c for c in CATEGORY if c in X_tr_enc.columns]
    X_tr_enc.drop(columns=cols_to_drop, inplace=True)
    X_va_enc.drop(columns=cols_to_drop, inplace=True)
    X_te_enc.drop(columns=cols_to_drop, inplace=True)

    X_tr_enc = reduce_mem_usage(X_tr_enc)
    X_va_enc = reduce_mem_usage(X_va_enc)
    X_te_enc = reduce_mem_usage(X_te_enc)

    # Base XGB
    print('Training XGB-Base...')
    model_base = XGBClassifier(**xgb_base_params)
    model_base.fit(X_tr_enc, y_tr_enc, sample_weight=w_tr_enc, verbose=False)

    va_p_base = model_base.predict_proba(X_va_enc)
    oof_xgb_base[va_idx] = va_p_base
    test_xgb_base += model_base.predict_proba(X_te_enc) / n_folds
    print(f"  Base Fold {fold} Balanced Accuracy: {balanced_accuracy_score(y_va, va_p_base.argmax(axis=1)):.6f}")

    del model_base
    gc.collect()

    # Variant-A XGB
    print('Training XGB-Variant-A...')
    X_tr_a = build_xgb_variant_a(X_tr_enc)
    X_va_a = build_xgb_variant_a(X_va_enc)
    X_te_a = build_xgb_variant_a(X_te_enc)

    model_a = XGBClassifier(**xgb_a_params)
    model_a.fit(X_tr_a, y_tr_enc, sample_weight=w_tr_enc, verbose=False)

    va_p_a = model_a.predict_proba(X_va_a)
    oof_xgb_a[va_idx] = va_p_a
    test_xgb_a += model_a.predict_proba(X_te_a) / n_folds
    print(f"  VarA Fold {fold} Balanced Accuracy: {balanced_accuracy_score(y_va, va_p_a.argmax(axis=1)):.6f}")

    del model_a, X_tr_a, X_va_a, X_te_a
    gc.collect()

    # Variant-B XGB
    print('Training XGB-Variant-B...')
    X_tr_b = build_xgb_variant_b(X_tr_enc)
    X_va_b = build_xgb_variant_b(X_va_enc)
    X_te_b = build_xgb_variant_b(X_te_enc)

    model_b = XGBClassifier(**xgb_b_params)
    model_b.fit(X_tr_b, y_tr_enc, sample_weight=w_tr_enc, verbose=False)

    va_p_b = model_b.predict_proba(X_va_b)
    oof_xgb_b[va_idx] = va_p_b
    test_xgb_b += model_b.predict_proba(X_te_b) / n_folds
    print(f"  VarB Fold {fold} Balanced Accuracy: {balanced_accuracy_score(y_va, va_p_b.argmax(axis=1)):.6f}")

    del model_b
    del X_tr_b, X_va_b, X_te_b
    del X_tr_enc, y_tr_enc, w_tr_enc, X_va_enc, X_te_enc, te_train
    gc.collect()


def score(p):
    return balanced_accuracy_score(y, p.argmax(axis=1))

print('CV Scores before blending:')
print('XGB Base CV:', round(score(oof_xgb_base), 6))
print('XGB VarA CV:', round(score(oof_xgb_a), 6))
print('XGB VarB CV:', round(score(oof_xgb_b), 6))


In [ ]:
#Blendign& submission Export (XGB trio)

import optuna
from optuna.samplers import TPESampler


def objective_blend(trial):
    # Model Weights: base + variant A + variant B
    w_base = trial.suggest_float('w_base', 0.0, 1.0)
    w_a = trial.suggest_float('w_a', 0.0, 1.0)
    w_b = trial.suggest_float('w_b', 0.0, 1.0)

    # Class-weights multipliers
    cw0 = trial.suggest_float('cw0', 0.5, 3.0)
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)

    w_sum = w_base + w_a + w_b
    if w_sum == 0:
        return 0.0

    w_base /= w_sum
    w_a /= w_sum
    w_b /= w_sum

    blend = w_base * oof_xgb_base + w_a * oof_xgb_a + w_b * oof_xgb_b

    cws = np.array([cw0, cw1, cw2])
    blend = blend * cws
    blend = blend / blend.sum(axis=1, keepdims=True)

    return balanced_accuracy_score(y, blend.argmax(axis=1))

oof_xgb_base = np.load("/kaggle/input/datasets/kashifalikhan360/oof-generated/oof_xgb_base.npy")
oof_xgb_a = np.load("/kaggle/input/datasets/kashifalikhan360/oof-generated/oof_xgb_a.npy")
oof_xgb_b = np.load("/kaggle/input/datasets/kashifalikhan360/oof-generated/oof_xgb_b.npy")

test_xgb_base = np.load("/kaggle/input/datasets/kashifalikhan360/test-oof-gen/test_xgb_base.npy")
test_xgb_a = np.load("/kaggle/input/datasets/kashifalikhan360/test-oof-gen/test_xgb_a.npy")
test_xgb_b = np.load("/kaggle/input/datasets/kashifalikhan360/test-oof-gen/test_xgb_b.npy")


# np.save("test_xgb_base.npy", test_xgb_base)
# np.save("test_xgb_a.npy", test_xgb_a)
# np.save("test_xgb_b.npy", test_xgb_b)

# study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
# study.optimize(objective_blend, n_trials=200, show_progress_bar=False)

# print('Best Blend CV Score:', round(study.best_value, 6))
# p = study.best_params
# print('Best Params:', p)

p={'w_base': 0.7374727840955072, 'w_a': 0.1224472873187584, 'w_b': 0.12542611519172017, 'cw0': 1.5524419482938487, 'cw1': 1.4563634062960278, 'cw2': 2.0183432146351143}
# p is the best weights i gto on 138 trial

# w_sum = p['w_base'] + p['w_a'] + p['w_b']
# best_model_w = np.array([p['w_base'] / w_sum, p['w_a'] / w_sum, p['w_b'] / w_sum])
# best_cws = np.array([p['cw0'], p['cw1'], p['cw2']])

# final_test_probs = (
#     best_model_w[0] * test_xgb_base
#     + best_model_w[1] * test_xgb_a
#     + best_model_w[2] * test_xgb_b
# )

# final_test_probs = final_test_probs * best_cws
# final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)

# final_test_idx = np.argmax(final_test_probs, axis=1)
# final_test_labels = pd.Series(final_test_idx).map(idx2label)

# submission = sub.copy()
# submission[ID_COL] = test_ids.values
# submission[TARGET_COL] = final_test_labels.values

#importing directly to submit fast
submission=pd.read_csv("submission (34).csv")
# submission.to_csv('/kaggle/working/submission.csv', index=False)
submission.head()